# FOD tespiti: Colab defteri

Bu defter veriyi hazırlar ve kontrol eder. Eğitim hücreleri sonraki adımda eklenecek.

**Başlamadan önce:**
1. Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU* seç.
2. Veri zip'i Drive'ım klasöründe doğrudan durmuyorsa ayar hücresindeki `ZIP_PATH` değerini düzelt.
3. Hücreleri yukarıdan aşağıya sırayla çalıştır.

In [ ]:
# AYARLAR: zip Drive'da başka bir yerdeyse ZIP_PATH satırını düzelt
REPO_URL = "https://github.com/ogutalanummugulsum-netizen/fod-tespit.git"  # GitHub depon
ZIP_PATH = "/content/drive/MyDrive/FODPascalVOCFormat-V.2.1.zip"  # Drive'daki veri zip'i

# Bunlara dokunmana gerek yok
PROJECT_DIR = "/content/proje1"  # kodun Colab'da duracağı yer
DRIVE_OUT = "/content/drive/MyDrive/fod_proje"  # sonuçların Drive'da saklanacağı klasör
VOC_DIR = "data/raw/FODPascalVOCFormat-V.2.1/VOC2007"

In [ ]:
# GPU açık mı?
import torch

if torch.cuda.is_available():
    print("GPU hazır:", torch.cuda.get_device_name(0))
else:
    print("GPU YOK. Veri hazırlama için sorun değil, ama eğitimden önce GPU'yu aç.")

In [ ]:
# Google Drive'ı bağla (izin penceresi açılır)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Kodu GitHub'dan çek ve paketleri kur
import os

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Veri zip'ini Drive'dan Colab diskine aç (Drive'dan doğrudan okumak çok yavaş)
if not os.path.exists(VOC_DIR):
    os.makedirs("data/raw", exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d data/raw

!ls {VOC_DIR}

## 1. Etiketleri YOLO formatına çevir

In [ ]:
!python scripts/convert_voc_to_yolo.py --voc-dir {VOC_DIR} --out-dir data/yolo

In [ ]:
# KONTROL: dönüşüm beklenen sayıları veriyor mu?
# Beklenen sayılar, ham veri ayrı bir yöntemle sayılarak bulundu.
from pathlib import Path

import pandas as pd

boxes = pd.read_csv("data/yolo/boxes.csv", dtype={"image_id": str})
class_counts = boxes["class_name"].value_counts()

checks = [
    ("Görüntü sayısı", boxes["image_id"].nunique(), 33793),
    ("Kutu sayısı", len(boxes), 34472),
    ("Sınıf sayısı", boxes["class_name"].nunique(), 31),
    ("Bolt kutusu", int(class_counts.get("Bolt", 0)), 3300),
    ("Screw kutusu", int(class_counts.get("Screw", 0)), 157),
    ("Tape kutusu", int(class_counts.get("Tape", 0)), 127),
    ("Tek sınıflı etiket dosyası", len(list(Path("data/yolo/all/labels_fod").glob("*.txt"))), 33793),
    ("31 sınıflı etiket dosyası", len(list(Path("data/yolo/all/labels_31").glob("*.txt"))), 33793),
    ("Koordinatlar 0-1 arasında", bool(boxes[["x_center", "y_center", "width", "height"]].stack().between(0, 1).all()), True),
]

all_ok = True
for name, found, expected in checks:
    ok = found == expected
    all_ok = all_ok and ok
    print(f"{'OK  ' if ok else 'HATA'}  {name}: bulunan {found}, beklenen {expected}")

assert all_ok, "Dönüşüm beklenen sayıları vermedi. Devam etme, önce nedenini bul."
print("\nDönüşüm doğru görünüyor.")

## 2. Train / val / test bölmesi

In [ ]:
!python scripts/make_splits.py --voc-dir {VOC_DIR} --data-dir data/yolo

In [ ]:
# KONTROL: bölme kurallara uyuyor mu? (script'ten bağımsız olarak yeniden hesaplanır)
splits = pd.read_csv("configs/splits.csv")
image_splits = pd.read_csv("data/yolo/image_splits.csv", dtype={"image_id": str})
df = boxes.merge(image_splits, on="image_id")
df["kucuk"] = df[["xmax", "ymax"]].sub(df[["xmin", "ymin"]].values).max(axis=1) < 32

total = int(splits["n_images"].sum())
frames = splits.groupby("split")["n_images"].agg(["sum", "max", "count"])
# Bir sınıfı içeren kare sayısı (bölüm başına)
class_frames = df.drop_duplicates(["image_id", "class_name"]).groupby(["split", "class_name"]).size()
small_share = 100 * df.groupby("split")["kucuk"].mean()
test_classes = set(class_frames["test"].index)
train_classes = set(class_frames["train"].index)
unseen = sorted(test_classes - train_classes)
val_top = class_frames["val"] / frames.loc["val", "sum"]

checks = [("Video parçası sayısı", len(splits), 116), ("Toplam kare", total, 33793)]
for name in ["val", "test"]:
    ratio = frames.loc[name, "sum"] / total
    share = frames.loc[name, "max"] / frames.loc[name, "sum"]
    diff = small_share[name] - small_share["train"]
    checks.append((f"{name} oranı %9-%11 arasında (bulunan %{100 * ratio:.1f})", bool(0.09 <= ratio <= 0.11), True))
    checks.append((f"{name} içinde en büyük parça yarıdan az (bulunan %{100 * share:.1f})", bool(share <= 0.5), True))
    checks.append((f"{name}: 32 pikselden küçük kutu oranı train'e yakın (fark {diff:+.1f} puan)", bool(abs(diff) <= 5), True))
for name in ["Bolt", "Nut", "Rock"]:
    checks.append((f"test'te {name} var", name in test_classes, True))
checks.append((f"val'de en az 12 parça (bulunan {frames.loc['val', 'count']})", bool(frames.loc["val", "count"] >= 12), True))
checks.append((f"val'de tek sınıf en fazla %25 (en büyük: {val_top.idxmax()} %{100 * val_top.max():.1f})", bool(val_top.max() <= 0.25), True))
checks.append((f"test'te eğitimde olmayan 2-4 sınıf (bulunan: {', '.join(unseen) or '-'})", 2 <= len(unseen) <= 4, True))

all_ok = True
for name, found, expected in checks:
    ok = found == expected
    all_ok = all_ok and ok
    print(f"{'OK  ' if ok else 'HATA'}  {name}: bulunan {found}, beklenen {expected}")

assert all_ok, "Bölme kurallara uymuyor. Devam etme."
print("\nKüçük kutu oranı (%):")
print(small_share.round(1))
splits[splits["split"] != "train"]

In [ ]:
# Sızıntı kontrolü (birkaç dakika sürebilir)
!python scripts/check_splits.py --data-dir data/yolo

## 3. Kutu boyutu raporu
Cisimler görüntünün ne kadarını kaplıyor? Gerçek pist kamerasında cisimler çok küçük görünür; bu tablo veri setinin buna ne kadar benzediğini gösterir.

In [ ]:
image_splits = pd.read_csv("data/yolo/image_splits.csv", dtype={"image_id": str})
df = boxes.merge(image_splits, on="image_id")

# Kutunun görüntü alanına oranı (%) ve uzun kenarı (piksel)
df["alan_yuzde"] = 100 * df["width"] * df["height"]
df["uzun_kenar_px"] = df[["width", "height"]].max(axis=1) * df["img_w"]

print("Kutunun görüntü alanına oranı (%):")
print(df.groupby("split")["alan_yuzde"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))

df["boyut"] = pd.cut(
    df["uzun_kenar_px"],
    bins=[0, 16, 32, 64, 96, 10000],
    labels=["<16 px", "16-32 px", "32-64 px", "64-96 px", ">=96 px"],
    right=False,
)
size_table = (pd.crosstab(df["boyut"], df["split"], normalize="columns") * 100).round(1)
print("\nKutunun uzun kenarına göre dağılım (her bölümdeki kutuların yüzdesi):")
print(size_table)

os.makedirs("results", exist_ok=True)
size_table.to_csv("results/box_size_report.csv")

## 4. Çıktıları Drive'a kaydet
Colab oturumu kapanınca `/content` içindeki her şey silinir. Bölme kaydı ve raporlar Drive'a kopyalanır.

In [ ]:
import shutil

os.makedirs(f"{DRIVE_OUT}/configs", exist_ok=True)
shutil.copy("configs/splits.csv", f"{DRIVE_OUT}/configs/splits.csv")
# Yalnızca veri hazırlama raporları kopyalanır; Drive'daki deney sonuçlarına dokunulmaz
os.makedirs(f"{DRIVE_OUT}/results", exist_ok=True)
for report in ["split_report.txt", "split_check.txt", "box_size_report.csv"]:
    shutil.copy(f"results/{report}", f"{DRIVE_OUT}/results/{report}")
print("Kaydedildi:", DRIVE_OUT)

## Sıradaki adım
Eğitim ve değerlendirme hücreleri bir sonraki adımda eklenecek.